# WeedBot weed detector - step 1 of 2: collect the training photos
Downloads ~12,000 public photos (weeds + plants to keep), converts them to two classes (`weed`, `keep`) and
stores the result in your Google Drive (`MyDrive/weedbot/dataset_parts/`, about 3 GB).

**How:** Runtime -> Change runtime type -> **CPU** is enough here (keeps your free GPU time for step 2),
then Runtime -> **Run all**. Allow Google Drive access when asked. Takes roughly 30-90 minutes.
If it stops, just press **Run all** again: finished parts are skipped.

Sources and licences: CottonWeedDet12 (CC BY-NC 4.0), Roboflow-100 grass-weeds (CC BY 4.0),
CropWeeds-YOLO (MIT), Open Images V7 (CC BY 2.0 images / CC BY 4.0 boxes).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE = '/content/drive/MyDrive/weedbot'
os.makedirs(DRIVE, exist_ok=True)
sys.path.insert(0, '/content')

In [ ]:
import os
os.makedirs("/content/weed_detector", exist_ok=True)

In [ ]:
%%writefile /content/weed_detector/__init__.py
"""WeedBot weed/keep detector: dataset building (sources, build, colab) and training (train)."""

In [ ]:
%%writefile /content/weed_detector/sources.py
"""sources.py - open weed / plant datasets -> items for ONE two-class YOLO dataset (weed = 0, keep = 1).

Why: the WeedBot detector learns from public photos only (the owner takes no photos). Each source has its
own labels; this file maps them onto two classes. Design: docs/superpowers/specs/2026-11-24-weed-detector-design.md.

  cottonweed   CottonWeedDet12 via Hugging Face Voxel51/CottonWeedDet12 (samples.json, FiftyOne boxes):
               all 12 weed species -> weed, EXCEPT grass-type weeds (Goosegrass): their boxes are dropped,
               because lawn grass is background and a broadleaf spray does not kill grasses.
  grassweeds   LibreYOLO/grass-weeds (YOLO, class "ridderzuring" = broad-leaved dock in grassland) -> weed.
  cropsweeds   toolambr/crops_weeds (YOLO, classes crop / weed) -> keep / weed.
  openimages   Open Images V7 (FiftyOne), Flower / Houseplant / Plant / Rose / Lily boxes -> keep; images
               that show a dandelion (a weed with a flower) or only drawings are skipped.

An Item is one image: where to get it (`src`, a local path or URL), its split and its YOLO boxes
(class, cx, cy, w, h, all relative 0..1).
"""
import glob
import hashlib
import json
import os
from collections import namedtuple

WEED, KEEP = 0, 1
CLASS_NAMES = ['weed', 'keep']
GRASS_TYPE = {'goosegrass', 'crabgrass', 'barnyardgrass', 'johnsongrass', 'foxtail', 'nutsedge'}
OI_KEEP = {'Flower', 'Houseplant', 'Plant', 'Rose', 'Lily'}
OI_WEED_HINTS = {'Dandelion'}
# Image-level labels (full Open Images vocabulary, human-verified) of weeds: a picture showing any of these
# never becomes a `keep` example. FiftyOne only loads the boxable labels, so these come from the separate
# `*-annotations-human-imagelabels.csv` files, see oi_weed_image_ids().
OI_WEED_MIDS = {'/m/0fr19': 'Dandelion', '/m/02qpvtx': 'Common Dandelion', '/m/0c2pc': 'Clover',
                '/m/0grfz': 'Spurge', '/m/06tzj3': 'Thistle', '/m/08486': 'Weed'}
OI_IMAGELABEL_URLS = ['https://storage.googleapis.com/openimages/v5/validation-annotations-human-imagelabels.csv',
                      'https://storage.googleapis.com/openimages/v5/test-annotations-human-imagelabels.csv']
OI_PEOPLE = {'Person', 'Man', 'Woman', 'Boy', 'Girl', 'Human body', 'Human face'}
OI_MIN_AREA = 0.005       # keep boxes smaller than 0.5 % of the image are too small to teach anything

LICENCES = {
    'cottonweed': 'CC BY-NC 4.0 - CottonWeedDet12, Lu et al., Zenodo 10.5281/zenodo.7535814 (HF Voxel51/CottonWeedDet12)',
    'grassweeds': 'CC BY 4.0 - Roboflow 100 "grass-weeds" (HF LibreYOLO/grass-weeds)',
    'cropsweeds': 'MIT - CropWeeds-YOLO (Kaggle swish9/weeds-detection) via HF toolambr/crops_weeds',
    'openimages': 'images CC BY 2.0, boxes CC BY 4.0 - Open Images V7 (Google)',
}

Item = namedtuple('Item', 'source name src split boxes licence orig')
SPLIT_DIRS = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'test'}


def fo_box_to_yolo(x, y, w, h):
    """FiftyOne / Open Images box [x, y, w, h] (relative, top-left) -> YOLO (cx, cy, w, h), clamped to the image."""
    x0, y0 = max(0.0, x), max(0.0, y)
    x1, y1 = min(1.0, x + w), min(1.0, y + h)
    return ((x0 + x1) / 2.0, (y0 + y1) / 2.0, x1 - x0, y1 - y0)


def stable_split(name, seed='weedbot'):
    """train / val / test (80 / 10 / 10) from a hash of the file name: the same image always lands in the same split."""
    h = int(hashlib.sha1(f'{seed}:{name}'.encode()).hexdigest()[:8], 16) % 100
    return 'train' if h < 80 else ('val' if h < 90 else 'test')


def cottonweed_items(samples_json, image_root):
    """CottonWeedDet12 samples.json -> Items. `image_root` is a folder or the HF base URL that holds data/*.jpg."""
    with open(samples_json) as f:
        samples = json.load(f)['samples']
    items = []
    for s in samples:
        name = os.path.basename(s['filepath'])
        boxes, orig = [], []
        for det in ((s.get('ground_truth') or {}).get('detections') or []):
            orig.append(det['label'])
            if det['label'].lower() in GRASS_TYPE:
                continue
            boxes.append((WEED,) + fo_box_to_yolo(*det['bounding_box']))
        src = image_root.rstrip('/') + '/' + s['filepath']
        items.append(Item('cottonweed', name, src, stable_split(name), boxes, LICENCES['cottonweed'], ';'.join(orig)))
    return items


def _yolo_line_to_box(parts):
    """'cls cx cy w h' or a polygon 'cls x1 y1 x2 y2 ...' -> (cx, cy, w, h)."""
    vals = [float(v) for v in parts[1:]]
    if len(vals) == 4:
        return tuple(vals)
    xs, ys = vals[0::2], vals[1::2]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    return ((x0 + x1) / 2.0, (y0 + y1) / 2.0, x1 - x0, y1 - y0)


def yolo_dir_items(root, source, names, class_map, licence):
    """A YOLO dataset folder (<split>/images/... or images/<split>/...) -> Items.
    `names` = the source's class names by id; `class_map` = name -> WEED / KEEP / None (None drops the box)."""
    root = str(root)
    items = []
    for img in sorted(glob.glob(os.path.join(root, '**', '*'), recursive=True)):
        if not img.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        rel = os.path.relpath(img, root).split(os.sep)
        if 'images' not in rel:
            continue
        split = next((SPLIT_DIRS[p] for p in rel if p in SPLIT_DIRS), None) or stable_split(rel[-1])
        lab_parts = list(rel)
        lab_parts[lab_parts.index('images')] = 'labels'
        lab = os.path.splitext(os.path.join(root, *lab_parts))[0] + '.txt'
        boxes, orig = [], []
        if os.path.exists(lab):
            with open(lab) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) < 5:
                        continue
                    cname = names[int(float(parts[0]))]
                    orig.append(cname)
                    cls = class_map.get(cname)
                    if cls is not None:
                        boxes.append((cls,) + _yolo_line_to_box(parts))
        items.append(Item(source, rel[-1], img, split, boxes, licence, ';'.join(orig)))
    return items


def oi_weed_image_ids(csv_paths):
    """Open Images image ids that a human tagged with a weed label (OI_WEED_MIDS, confidence 1)."""
    import csv
    ids = set()
    for path in csv_paths:
        with open(path) as f:
            for r in csv.DictReader(f):
                if r['LabelName'] in OI_WEED_MIDS and r['Confidence'].startswith('1'):
                    ids.add(r['ImageID'])
    return ids


def openimages_items(dataset, max_images=3000, det_field='detections', exclude_ids=frozenset()):
    """FiftyOne Open Images samples -> Items with KEEP boxes only. Skips drawings (IsDepiction), images
    tagged with a weed (dandelion) or with people, images whose keep box covers a GROUP of plants
    (IsGroupOf), tiny keep boxes, and images left without any keep box. FiftyOne's Open Images zoo stores the
    boxes in the field `detections` (image-level labels in `positive_labels`)."""
    items = []
    for s in dataset:
        positives = set()
        if s.has_field('positive_labels') and s['positive_labels'] is not None:
            positives = {c.label for c in s['positive_labels'].classifications}
        if positives & (OI_WEED_HINTS | OI_PEOPLE):
            continue                                    # a weed flower in view, or a people scene
        if os.path.splitext(os.path.basename(s.filepath))[0] in exclude_ids:
            continue                                    # tagged with a weed (dandelion, clover, spurge...)
        dets = s[det_field].detections if s.has_field(det_field) and s[det_field] is not None else []
        if any(getattr(d, 'IsDepiction', False) for d in dets):
            continue                                    # drawings, not real plants
        keep_dets = [d for d in dets if d.label in OI_KEEP]
        if any(getattr(d, 'IsGroupOf', False) for d in keep_dets):
            continue                                    # one box over a whole hedge / bed: would teach "patch = keep"
        boxes = [(KEEP,) + fo_box_to_yolo(*d.bounding_box) for d in keep_dets
                 if d.bounding_box[2] * d.bounding_box[3] >= OI_MIN_AREA]
        if not boxes:
            continue
        name = os.path.basename(s.filepath)
        items.append(Item('openimages', name, s.filepath, stable_split(name), boxes, LICENCES['openimages'],
                          ';'.join(sorted({d.label for d in dets}))))
        if len(items) >= max_images:
            break
    return items

In [ ]:
%%writefile /content/weed_detector/build.py
"""build.py - write Items (sources.py) as a YOLO dataset: resized images, label files, manifest, data.yaml.

Layout:  <out>/{train,val,test}/{images,labels}/<source>__<name>.jpg|.txt, <out>/manifest.csv,
<out>/data.yaml. Resumable: an image already written is not redone, and build_source() skips a source
whose `.done_<source>` marker exists (a Colab session can stop and be re-run).
"""
import csv
import io
import os
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor

from PIL import Image, ImageOps

from .sources import CLASS_NAMES

MANIFEST_FIELDS = ['file', 'split', 'source', 'n_boxes', 'orig_labels', 'licence']


def _retry_wait(err, attempt):
    """Seconds to wait before the next try: the server's Retry-After for 429 / 503, else 2, 4, 8 ... (max 60)."""
    import urllib.error
    if isinstance(err, urllib.error.HTTPError) and err.code in (429, 503):
        try:
            return min(120.0, float(err.headers.get('Retry-After', '')))
        except (TypeError, ValueError):
            return min(60.0, 5.0 * 2 ** attempt)
    return min(60.0, 2.0 * 2 ** attempt)


def fetch(src, retries=8, sleep=time.sleep):
    """Bytes of a local file or a URL. Retries with growing waits: Hugging Face answers 429 ("too many
    requests") to anonymous downloads in bursts, and big downloads sometimes drop."""
    if not src.startswith(('http://', 'https://')):
        with open(src, 'rb') as f:
            return f.read()
    for attempt in range(retries):
        try:
            req = urllib.request.Request(src, headers={'User-Agent': 'weedbot-dataset-builder'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except Exception as e:
            if attempt == retries - 1:
                raise
            sleep(_retry_wait(e, attempt))


def _out_paths(out_dir, item):
    stem = f'{item.source}__{os.path.splitext(item.name)[0]}'
    img = os.path.join(out_dir, item.split, 'images', stem + '.jpg')
    lab = os.path.join(out_dir, item.split, 'labels', stem + '.txt')
    return img, lab


def _write_one(item, out_dir, long_side, quality):
    img_path, lab_path = _out_paths(out_dir, item)
    if os.path.exists(img_path) and os.path.exists(lab_path):
        return None
    with Image.open(io.BytesIO(fetch(item.src))) as im:
        im = ImageOps.exif_transpose(im).convert('RGB')
        scale = long_side / float(max(im.size))
        if scale < 1.0:
            im = im.resize((round(im.width * scale), round(im.height * scale)), Image.LANCZOS)
        os.makedirs(os.path.dirname(img_path), exist_ok=True)
        os.makedirs(os.path.dirname(lab_path), exist_ok=True)
        im.save(img_path + '.part', 'JPEG', quality=quality)
    with open(lab_path + '.part', 'w') as f:
        for c, cx, cy, w, h in item.boxes:
            f.write(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')
    os.replace(lab_path + '.part', lab_path)
    os.replace(img_path + '.part', img_path)          # the image last: it marks the pair as complete
    return {'file': os.path.relpath(img_path, out_dir), 'split': item.split, 'source': item.source,
            'n_boxes': len(item.boxes), 'orig_labels': item.orig, 'licence': item.licence}


def build(items, out_dir, long_side=1024, quality=90, workers=1, log_every=500):
    """Write every item not yet written. Returns {'written': n, 'skipped_existing': n, 'failed': n}."""
    out_dir = str(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    manifest = os.path.join(out_dir, 'manifest.csv')
    new_file = not os.path.exists(manifest)
    counts = {'written': 0, 'skipped_existing': 0, 'failed': 0}
    with open(manifest, 'a', newline='') as mf:
        writer = csv.DictWriter(mf, fieldnames=MANIFEST_FIELDS)
        if new_file:
            writer.writeheader()

        def job(it):
            try:
                return _write_one(it, out_dir, long_side, quality)
            except Exception as e:                      # one bad download must not stop 10,000 images
                return {'error': f'{it.source}/{it.name}: {e}'}

        with ThreadPoolExecutor(max_workers=workers) as pool:
            for i, row in enumerate(pool.map(job, items), 1):
                if row is None:
                    counts['skipped_existing'] += 1
                elif 'error' in row:
                    counts['failed'] += 1
                    print('FAILED', row['error'])
                else:
                    writer.writerow(row)
                    counts['written'] += 1
                if log_every and i % log_every == 0:
                    mf.flush()
                    print(f'{i}/{len(items)} {counts}', flush=True)
    return counts


def build_source(source, items, out_dir, **kw):
    """build() one source and leave a `.done_<source>` marker; a finished source is skipped next time."""
    marker = os.path.join(str(out_dir), f'.done_{source}')
    if os.path.exists(marker):
        return {'skipped': True}
    counts = build(items, out_dir, **kw)
    if counts['failed'] == 0:
        with open(marker, 'w') as f:
            f.write(str(counts))
    return counts


def write_yaml(out_dir):
    """data.yaml for Ultralytics (paths relative to `path`)."""
    p = os.path.join(str(out_dir), 'data.yaml')
    names = '\n'.join(f'  {i}: {n}' for i, n in enumerate(CLASS_NAMES))
    with open(p, 'w') as f:
        f.write(f'path: {os.path.abspath(str(out_dir))}\ntrain: train/images\nval: val/images\ntest: test/images\n'
                f'names:\n{names}\n')
    from pathlib import Path
    return Path(p)

In [ ]:
%%writefile /content/weed_detector/colab.py
"""colab.py - the glue the two Colab notebooks run: one dataset part per source on Google Drive, merging the
parts, and the source downloads. Kept here (not only in the notebooks) so it can be tested on the VM.

Drive layout (MyDrive/weedbot/):  dataset_parts/<source>.zip   one finished source each (a stopped
session only redoes the unfinished source);  runs/v1/   training checkpoints;  weed_detector_v1.zip  result.
"""
import collections
import glob
import os
import shutil
import zipfile

from . import build as B
from . import sources as S

HF = 'https://huggingface.co/datasets'


def part_zip(drive_dir, source):
    return os.path.join(drive_dir, 'dataset_parts', f'{source}.zip')


def build_part(source, items_fn, work_dir, drive_dir, **build_kw):
    """Build one source into work_dir/<source> and store it as dataset_parts/<source>.zip on Drive.
    `items_fn()` is only called (downloads only start) if that zip does not exist yet."""
    target = part_zip(drive_dir, source)
    if os.path.exists(target):
        print(f'{source}: already on Drive, skipped')
        return {'skipped': True}
    out = os.path.join(work_dir, source)
    counts = B.build_source(source, items_fn(), out, **build_kw)
    if counts.get('failed'):
        print(f'{source}: {counts["failed"]} images failed; run this cell again to retry them')
        return counts
    man = os.path.join(out, 'manifest.csv')
    if os.path.exists(man):
        os.replace(man, os.path.join(out, f'manifest_{source}.csv'))   # one manifest per part, no clash on merge
    os.makedirs(os.path.dirname(target), exist_ok=True)
    tmp = shutil.make_archive(os.path.join(work_dir, f'{source}_part'), 'zip', out)
    shutil.copy(tmp, target + '.part')
    os.replace(target + '.part', target)
    print(f'{source}: {counts} -> {target}')
    return counts


def merge_parts(drive_dir, out_dir):
    """Unzip every finished part into one dataset folder and write data.yaml. Returns summary(out_dir)."""
    os.makedirs(out_dir, exist_ok=True)
    for z in sorted(glob.glob(os.path.join(drive_dir, 'dataset_parts', '*.zip'))):
        with zipfile.ZipFile(z) as f:
            f.extractall(out_dir)
    B.write_yaml(out_dir)
    return summary(out_dir)


def summary(out_dir):
    """{split: {source: images}} and {class name: boxes} from the label files."""
    images = collections.defaultdict(collections.Counter)
    boxes = collections.Counter()
    for split in ('train', 'val', 'test'):
        for lab in glob.glob(os.path.join(out_dir, split, 'labels', '*.txt')):
            images[split][os.path.basename(lab).split('__')[0]] += 1
            with open(lab) as f:
                for line in f:
                    if line.strip():
                        boxes[S.CLASS_NAMES[int(line.split()[0])]] += 1
    return {'images': {k: dict(v) for k, v in images.items()}, 'boxes': dict(boxes)}


# ---- source downloads (need internet; used in the Colab notebook) ----

def cottonweed_items_from_hf(tmp_dir):
    """CottonWeedDet12 labels file from Hugging Face; images are fetched one by one while building."""
    import urllib.request
    os.makedirs(tmp_dir, exist_ok=True)
    sj = os.path.join(tmp_dir, 'cottonweed_samples.json')
    if not os.path.exists(sj):
        urllib.request.urlretrieve(f'{HF}/Voxel51/CottonWeedDet12/resolve/main/samples.json', sj)
    return S.cottonweed_items(sj, f'{HF}/Voxel51/CottonWeedDet12/resolve/main')


def hf_yolo_items(repo, source, names, class_map, dest):
    """Download a (small) YOLO dataset from Hugging Face and convert it."""
    from huggingface_hub import snapshot_download
    snapshot_download(repo, repo_type='dataset', local_dir=dest)
    return S.yolo_dir_items(dest, source, names, class_map, S.LICENCES[source])


def openimages_items_from_zoo(tmp_dir, max_images=3000, max_samples_per_split=6000):
    """Open Images V7 validation + test pictures with plant boxes (FiftyOne), minus weed-tagged pictures."""
    import urllib.request
    import fiftyone.zoo as foz
    os.makedirs(tmp_dir, exist_ok=True)
    csvs = []
    for url in S.OI_IMAGELABEL_URLS:
        p = os.path.join(tmp_dir, os.path.basename(url))
        if not os.path.exists(p):
            urllib.request.urlretrieve(url, p)
        csvs.append(p)
    exclude = S.oi_weed_image_ids(csvs)
    print(f'openimages: {len(exclude)} pictures tagged with a weed are left out')
    items = []
    for split in ('validation', 'test'):
        ds = foz.load_zoo_dataset('open-images-v7', split=split, label_types=['detections', 'classifications'],
                                  classes=sorted(S.OI_KEEP), max_samples=max_samples_per_split,
                                  dataset_name=f'weedbot_oi_{split}', overwrite=True)
        items += S.openimages_items(ds, max_images=max_images - len(items), exclude_ids=exclude)
        if len(items) >= max_images:
            break
    return items

In [ ]:
%%writefile /content/weed_detector/train.py
"""train.py - train / resume / evaluate / export the WeedBot weed-and-keep detector (Ultralytics YOLO11n).

Why resumable: free Colab sessions stop after a few hours. Checkpoints live in `project_dir` (Google Drive
in Colab); calling train() again continues from `last.pt` instead of starting over.
"""
import csv
import json
import os
from pathlib import Path

from .sources import CLASS_NAMES

TRAIN_DEFAULTS = dict(epochs=100, imgsz=640, patience=30, hsv_v=0.5, hsv_s=0.6, degrees=10.0, flipud=0.5,
                      batch=-1, workers=2)   # flipud: the camera looks DOWN at the ground, so upside-down is fine


def run_dir(project_dir, name='v1'):
    return Path(project_dir) / name


def train(data_yaml, project_dir, name='v1', base='yolo11n.pt', **overrides):
    """Train from `base`, or resume from <project_dir>/<name>/weights/last.pt. Returns the best.pt path."""
    from ultralytics import YOLO
    rd = run_dir(project_dir, name)
    last, best = rd / 'weights' / 'last.pt', rd / 'weights' / 'best.pt'
    if last.exists():
        try:
            YOLO(str(last)).train(resume=True)
        except AssertionError as e:                 # "training to N epochs is finished, nothing to resume"
            print('resume:', e)
        return best
    args = dict(TRAIN_DEFAULTS)
    args.update(overrides)
    YOLO(base).train(data=str(data_yaml), project=str(project_dir), name=name, exist_ok=True, **args)
    return best


def _source_lists(data_dir, split):
    """{source: [image paths]} for one split, from the `<source>__` file-name prefix."""
    out = {}
    idir = Path(data_dir) / split / 'images'
    for p in sorted(idir.glob('*.jpg')):
        out.setdefault(p.name.split('__')[0], []).append(str(p))
    return out


def evaluate_per_source(best_pt, data_dir, split='test', **val_kw):
    """mAP50 per class, precision and recall for each source's images in `split`."""
    from ultralytics import YOLO
    model = YOLO(str(best_pt))
    results = {}
    tmp = Path(data_dir) / '_eval'
    tmp.mkdir(exist_ok=True)
    for source, paths in _source_lists(data_dir, split).items():
        lst = tmp / f'{source}_{split}.txt'
        lst.write_text('\n'.join(paths) + '\n')
        yml = tmp / f'{source}_{split}.yaml'
        names = ''.join(f'  {i}: {n}\n' for i, n in enumerate(CLASS_NAMES))
        yml.write_text(f'path: {Path(data_dir).resolve()}\ntrain: {lst}\nval: {lst}\ntest: {lst}\nnames:\n{names}')
        m = model.val(data=str(yml), split='val', plots=False, verbose=False, **val_kw)
        ap50 = {CLASS_NAMES[int(c)]: round(float(v), 3) for c, v in zip(m.box.ap_class_index, m.box.ap50)}
        results[source] = {'images': len(paths), 'map50': ap50, 'precision': round(float(m.box.mp), 3),
                           'recall': round(float(m.box.mr), 3)}
    return results


def export(best_pt, imgsz=640, formats=('onnx', 'ncnn')):
    """Export for the Pi: ONNX and NCNN (the Pi 4 runs NCNN fastest on its CPU)."""
    from ultralytics import YOLO
    return [YOLO(str(best_pt)).export(format=f, imgsz=imgsz) for f in formats]


def summary(project_dir, name='v1'):
    """Last row of results.csv (per-epoch metrics) as a dict, for a quick look."""
    p = run_dir(project_dir, name) / 'results.csv'
    if not p.exists():
        return {}
    rows = list(csv.DictReader(open(p)))
    return {k.strip(): v for k, v in rows[-1].items()} if rows else {}


if __name__ == '__main__':
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument('data_yaml'); ap.add_argument('project_dir'); ap.add_argument('--epochs', type=int, default=100)
    a = ap.parse_args()
    b = train(a.data_yaml, a.project_dir, epochs=a.epochs)
    print(json.dumps(evaluate_per_source(b, os.path.dirname(a.data_yaml)), indent=1))

In [ ]:
!pip -q install huggingface_hub fiftyone

In [ ]:
from weed_detector import colab as C, sources as S
WORK = '/content/work'
C.build_part('cottonweed', lambda: C.cottonweed_items_from_hf(WORK), WORK, DRIVE, workers=8)

In [ ]:
C.build_part('grassweeds', lambda: C.hf_yolo_items('LibreYOLO/grass-weeds', 'grassweeds', ['ridderzuring'],
             {'ridderzuring': S.WEED}, WORK + '/src_grassweeds'), WORK, DRIVE, workers=8)

In [ ]:
C.build_part('cropsweeds', lambda: C.hf_yolo_items('toolambr/crops_weeds', 'cropsweeds', ['crop', 'weed'],
             {'crop': S.KEEP, 'weed': S.WEED}, WORK + '/src_cropsweeds'), WORK, DRIVE, workers=8)

In [ ]:
C.build_part('openimages', lambda: C.openimages_items_from_zoo(WORK + '/oi', max_images=3000), WORK, DRIVE, workers=8)

In [ ]:
import glob
parts = sorted(glob.glob(DRIVE + '/dataset_parts/*.zip'))
for p in parts:
    print(os.path.basename(p), round(os.path.getsize(p) / 1e6), 'MB')
print('DONE: all 4 parts are on Drive, go to step 2' if len(parts) == 4 else 'Not all parts finished: Run all again')